# pandas — Cross-Sectional Crypto Feature Factory + Regime Study

## Project goal

Build a reproducible pipeline on `data/crypto_hourly.parquet` that produces (a) a clean per-symbol feature panel, (b) cross-sectional ranks at every timestamp, and (c) a Markdown table of average forward 24h return conditional on BTC's volatility regime × momentum-rank bucket.

End state: one notebook that runs end-to-end in <30 seconds.


## Why this exercises the cheatsheet

Crypto OHLCV is inherently a panel (time × symbol × feature). You can't avoid `groupby`, MultiIndex, alignment, and the long↔wide reshape — they're forced by the structure. The project hits 12 of 14 cheatsheet topics in one artifact.


## Sub-task 1: Continuity validation

Load the parquet and verify each symbol has a continuous hourly UTC index. Build the expected `pd.date_range`, reindex each symbol against it, and report the count of missing bars per symbol. **If a symbol has gaps, decide explicitly: drop it, fill it, or assert** — and write the decision in a comment.

**Patterns this forces:**

- `pd.date_range(start, end, freq='1h', tz='UTC')`
- `groupby('symbol')` then `set_index('ts').reindex(expected)`
- `.isna().sum()` per symbol


In [29]:
# Your answer here

import pandas as pd

# load
df = pd.read_parquet('./../../data/crypto_hourly.parquet')

df['ts'] = pd.to_datetime(df['ts'], utc=True) # convert timestamp to datetime
expected = pd.date_range(df['ts'].min(), df['ts'].max(), freq='1h', tz='UTC')

# pivot
wide = df.pivot(index='ts', columns='symbol', values='close').sort_index()

# number of gaps per symbol
print(f'Number of gaps per symbol:\n\n{wide.reindex(expected).isna().sum()}')

Number of gaps per symbol:

symbol
BNB    0
BTC    0
ETH    0
SOL    0
dtype: int64


In [39]:
df.groupby('symbol').apply(lambda t: t.set_index('ts').reindex(expected).isna().any(axis=1).sum())

/tmp/ipykernel_702993/3547211652.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('symbol').apply(lambda t: t.set_index('ts').reindex(expected).isna().any(axis=1).sum())


symbol
BNB    0
BTC    0
ETH    0
SOL    0
dtype: int64

## Sub-task 2: Per-symbol features

Compute per-symbol features: log returns at 1h/4h/24h, 24h and 168h rolling realised volatility, ATR(14), RSI(14), z-scored 24h volume. All features must be **strictly past-only** (shift before rolling). Store them as new columns on the long frame.

**Patterns this forces:**

- `groupby('symbol').rolling(...).transform`
- `.shift(1)` BEFORE any rolling aggregation
- method-chained `.assign(...)` for feature blocks
- vectorised `np.log` / `.diff()`


In [74]:
# Your answer here

import numpy as np

def rsi(close, period=14):
    diff = close.diff()
    gains = diff.clip(lower=0)
    losses = -diff.clip(upper=0)

    avg_gains = gains.ewm(alpha=1/period, adjust=False).mean()
    avg_losses = losses.ewm(alpha=1/period, adjust=False).mean()

    rs = avg_gains / avg_losses.replace(0, np.nan)

    return 100 - 100 / (1 + rs)

def atr(g, period=14):
    h, l, c = g['high'], g['low'], g['close']
    prev_c = c.shift(1)
    tr = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
  
    return tr.ewm(alpha=1/period, adjust=False).mean()

def add_features(df):
    features = df.copy()

    # log returns
    for h in [1, 4, 24]:
        features[f'logret_{h}h'] = features.groupby('symbol')['close'].transform(lambda c: np.log(c).diff(h))

    # realised vol
    for h in [24, 168]:
        features[f'vol_{h}h'] = features.groupby('symbol')['logret_1h'].transform(lambda r: r.shift(1).rolling(h).std())

    # rsi 14
    features['rsi_14'] = features.groupby('symbol')['close'].transform(lambda c: rsi(c, 14))

    # atr 14
    features['atr_14'] = (
        features.groupby('symbol').apply(atr, include_groups=False).reset_index(level=0, drop=True)
    )

    # zscore volume 24h
    features['volume_z_24h'] = features.groupby('symbol')['volume'].transform(lambda v: (v - v.shift(1).rolling(24).mean()) / v.shift(1).rolling(24).std())
    
    return features

features = add_features(df)
features.head()

,ts,open,high,low,close,volume,symbol,logret_1h,logret_4h,logret_24h,vol_24h,vol_168h,rsi_14,atr_14,volume_z_24h
0,2024-04-19 23:00:00+00:00,556.7,557.0,549.6,554.0,21953.020,BNB,NaN,NaN,NaN,NaN,NaN,NaN,7.400000,NaN
1,2024-04-20 00:00:00+00:00,554.1,557.1,551.2,551.4,14108.411,BNB,-0.004704,NaN,NaN,NaN,NaN,0.000000,7.292857,NaN
2,2024-04-20 01:00:00+00:00,551.3,557.4,548.8,556.9,12407.915,BNB,0.009925,NaN,NaN,NaN,NaN,13.994911,7.386224,NaN
3,2024-04-20 02:00:00+00:00,556.9,558.1,554.7,557.0,7299.961,BNB,0.000180,NaN,NaN,NaN,NaN,14.229943,7.101494,NaN
4,2024-04-20 03:00:00+00:00,557.1,558.1,555.6,556.5,5680.975,BNB,-0.000898,0.004502,NaN,NaN,NaN,14.023587,6.772816,NaN


In [75]:
features = (
    df
    .assign(logret_1h=lambda d: d.groupby('symbol')['close'].transform(lambda c: np.log(c).diff(1)))
    .assign(logret_4h=lambda d: d.groupby('symbol')['close'].transform(lambda c: np.log(c).diff(4)))
    .assign(logret_24h=lambda d: d.groupby('symbol')['close'].transform(lambda c: np.log(c).diff(24)))
    .assign(vol_24h=lambda d: d.groupby('symbol')['logret_1h'].transform(lambda r: r.shift(1).rolling(24).std()))
    .assign(vol_168h=lambda d: d.groupby('symbol')['logret_1h'].transform(lambda r: r.shift(1).rolling(168).std()))
    .assign(rsi_14=lambda d: d.groupby('symbol')['close'].transform(lambda c: rsi(c, 14)))
    .assign(atr_14=lambda d: (
        d.groupby('symbol')
            .apply(atr, include_groups=False)
            .reset_index(level=0, drop=True)
    ))
    .assign(volume_z_24h=lambda d: d.groupby('symbol')['volume'].transform(lambda v: (v - v.shift(1).rolling(24).mean()) / v.shift(1).rolling(24).std()))
)

features.head()

,ts,open,high,low,close,volume,symbol,logret_1h,logret_4h,logret_24h,vol_24h,vol_168h,rsi_14,atr_14,volume_z_24h
0,2024-04-19 23:00:00+00:00,556.7,557.0,549.6,554.0,21953.020,BNB,NaN,NaN,NaN,NaN,NaN,NaN,7.400000,NaN
1,2024-04-20 00:00:00+00:00,554.1,557.1,551.2,551.4,14108.411,BNB,-0.004704,NaN,NaN,NaN,NaN,0.000000,7.292857,NaN
2,2024-04-20 01:00:00+00:00,551.3,557.4,548.8,556.9,12407.915,BNB,0.009925,NaN,NaN,NaN,NaN,13.994911,7.386224,NaN
3,2024-04-20 02:00:00+00:00,556.9,558.1,554.7,557.0,7299.961,BNB,0.000180,NaN,NaN,NaN,NaN,14.229943,7.101494,NaN
4,2024-04-20 03:00:00+00:00,557.1,558.1,555.6,556.5,5680.975,BNB,-0.000898,0.004502,NaN,NaN,NaN,14.023587,6.772816,NaN


## Sub-task 3: Cross-sectional ranks per timestamp

At each timestamp, rank the 4 symbols by their trailing 24h log return. Use the percentile rank (`pct=True`) so ranks ∈ [0, 1]. Store as a new column `xs_rank_ret_24h`. Verify the rank within each timestamp sums to (n_assets × 0.5) — a sanity check that ranks are well-formed.

**Patterns this forces:**

- `groupby('ts')['log_ret_24h'].transform('rank', pct=True)`
- (or) `pivot` long→wide → `.rank(axis=1, pct=True)` → `melt` back
- the cross-sectional z-score idiom from the earlier discussion


In [104]:
# Your answer here

features['xs_rank_ret_24h'] = features.groupby('ts')['logret_24h'].transform('rank', pct=True)

features.head()

,ts,open,high,low,close,volume,symbol,logret_1h,logret_4h,logret_24h,vol_24h,vol_168h,rsi_14,atr_14,volume_z_24h,xs_rank_ret_24h
0,2024-04-19 23:00:00+00:00,556.7,557.0,549.6,554.0,21953.020,BNB,NaN,NaN,NaN,NaN,NaN,NaN,7.400000,NaN,NaN
1,2024-04-20 00:00:00+00:00,554.1,557.1,551.2,551.4,14108.411,BNB,-0.004704,NaN,NaN,NaN,NaN,0.000000,7.292857,NaN,NaN
2,2024-04-20 01:00:00+00:00,551.3,557.4,548.8,556.9,12407.915,BNB,0.009925,NaN,NaN,NaN,NaN,13.994911,7.386224,NaN,NaN
3,2024-04-20 02:00:00+00:00,556.9,558.1,554.7,557.0,7299.961,BNB,0.000180,NaN,NaN,NaN,NaN,14.229943,7.101494,NaN,NaN
4,2024-04-20 03:00:00+00:00,557.1,558.1,555.6,556.5,5680.975,BNB,-0.000898,0.004502,NaN,NaN,NaN,14.023587,6.772816,NaN,NaN


## Sub-task 4: Multi-horizon resample

Resample hourly bars to 4h and 1d OHLCV per symbol with the right aggregation per column: `open='first'`, `high='max'`, `low='min'`, `close='last'`, `volume='sum'`. Store the daily frame as `bars_1d`.

**Patterns this forces:**

- `groupby('symbol').resample('4h').agg({'open': 'first', ...})`
- explicit per-column agg (NOT `.agg('sum')` blanket)
- preserving the (symbol, ts) MultiIndex on output


In [121]:
# Your answer here

OHLCV_AGG = {
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
}

def to_higher_freq(df, freq):
    return (
        df.set_index('ts')
            .sort_index()
            .groupby('symbol')
            .resample(freq)
            .agg(OHLCV_AGG)
            .reset_index()
    )

ohlcv_4h = to_higher_freq(df, '4h')
ohlcv_1d = to_higher_freq(df, '1d')

ohlcv_1d

,symbol,ts,open,high,low,close,volume
0,BNB,2024-04-19 00:00:00+00:00,556.70,557.00,549.60,554.00,21953.020
1,BNB,2024-04-20 00:00:00+00:00,554.10,574.30,548.80,570.90,242132.859
2,BNB,2024-04-21 00:00:00+00:00,570.90,582.60,566.30,579.60,277755.724
3,BNB,2024-04-22 00:00:00+00:00,579.60,608.70,578.40,604.50,522892.157
4,BNB,2024-04-23 00:00:00+00:00,604.40,618.30,598.70,606.20,556883.344
...,...,...,...,...,...,...,...
2919,SOL,2026-04-15 00:00:00+00:00,83.72,85.83,82.65,84.90,2602448.593
2920,SOL,2026-04-16 00:00:00+00:00,84.90,90.53,83.80,89.05,3940291.341
2921,SOL,2026-04-17 00:00:00+00:00,89.06,90.73,87.34,88.76,4237208.388
2922,SOL,2026-04-18 00:00:00+00:00,88.77,89.16,85.78,86.16,1893190.543


## Sub-task 5: Regime tagging

Compute BTC's 7-day rolling realised vol on the daily bars from the previous step. Bin it into terciles using `qcut` and tag each bar with a regime label `low_vol` / `mid_vol` / `high_vol`. Convert to `category` dtype for memory + speed. Confirm the regime distribution is roughly equal.

**Patterns this forces:**

- `pd.qcut(series, q=3, labels=[...])`
- `.astype('category')`
- `value_counts(normalize=True)` to confirm ~33% per bucket


In [142]:
# Your answer here

REGIME_LABELS = ['low_vol', 'mid_vol', 'high_vol']

btc_regime = (
    ohlcv_1d
    .query('symbol == "BTC"').set_index('ts').sort_index()
    .assign(logret=lambda d: np.log(d['close']).diff())
    .assign(vol_7d=lambda d: d['logret'].shift(1).rolling(7).std())
    .assign(regime=lambda d: pd.qcut(d['vol_7d'], q=3, labels=REGIME_LABELS))
)

btc_regime['regime'].value_counts(normalize=True).round(3)

regime
low_vol     0.333
mid_vol     0.333
high_vol    0.333
Name: proportion, dtype: float64

## Sub-task 6: Conditional return analysis

For each (regime, momentum-rank bucket from sub-task 3), compute the **mean forward 24h log return**. Bucket the rank into N-tiles where **N = number of symbols** in the universe (here 4 symbols → `q=4`). With pct-rank only producing N distinct values per timestamp, asking for more bins than symbols collapses bin edges and `qcut` raises `ValueError: Bin edges must be unique`. Output a `pivot_table` with regime as rows and rank-bucket as columns; cell values are mean forward return.

**This is the deliverable — a small table that answers a real quant question.**

**Patterns this forces:**

- `.shift(-24)` (per symbol!) for the forward-looking target
- `pivot_table(values=..., index=..., columns=..., aggfunc='mean')`
- `merge` / `align` of features and BTC regime by date (regime is daily, features are hourly)
- `pd.qcut(..., q=n_symbols, labels=[...])` — match bin count to the cardinality of your input

In [173]:
# Your answer here

# 24h forward returns
features['fwd_ret_24h'] = (
    features.groupby('symbol')['close']
    .transform(lambda c: np.log(c.shift(-24) / c))
)

regime_by_date = btc_regime[['regime']].rename_axis('date')

def reporting_fn(df):
    return (
        df.pivot_table(
            index='regime',
            columns='rank_bucket',
            values='fwd_ret_24h',
            aggfunc='mean',
            observed=False,
        )
    )

table = (
    features
    .assign(date=lambda d: d['ts'].dt.normalize()) # create daily timestamp from hourly
    .merge(regime_by_date, on='date', how='left') # merge btc regime w/ features
    .assign(rank_bucket=lambda d: pd.qcut(d['xs_rank_ret_24h'], q=4, labels=[f'Q{i}' for i in range(1, 5)]))
    .pipe(reporting_fn)
)

print(f'Result:\n{table.round(4).to_string()}')

Result:
rank_bucket      Q1      Q2      Q3      Q4
regime                                     
low_vol     -0.0025 -0.0013 -0.0010  0.0003
mid_vol     -0.0004 -0.0002 -0.0003 -0.0004
high_vol     0.0017  0.0013  0.0007 -0.0003


#### low_vol
- Yesterday's winners outperform yesterday's losers, Q4 - Q1 = +28bps/day
#### med_vol
- Limited signal. All assets have a similar performance
#### high_vol
- Yesterday's losers outoerform yesterday's winners, Q1 - Q4 = +20 bps/day

## What success looks like

- The notebook runs top-to-bottom in <30s with no errors.
- Sub-task 1 reports 0 missing bars (or you've handled them with a comment).
- Sub-task 6 outputs a `3 × n_symbols` table (here 3×4) where each cell is a mean forward return.
- You can read off the table whether *high-vol regimes have momentum or mean-reversion* on this universe.
- You wrote ≤ 1 `apply(axis=1)` call (and that one truly couldn't be vectorised).